In [ ]:
# Instalamos/actualizamos lo necesario
!pip install -q datasets --upgrade
!pip install -q evaluate
!pip install -q wordcloud
!pip install -q accelerate>=0.21.0
!pip install -q torchvision --upgrade



⚙️ **Requerimientos importantes sobre el ejercicio**

- El notebook debe ejecutarse **de principio a fin sin intervención manual**.
- Si utilizas librerías que no están incluidas por defecto en Google Colab, **asegúrate de instalarlas dentro del notebook** (por ejemplo: `!pip install ...`).

- Algunas celdas incluyen identificadores especiales que indican ciertas normas que **debes** respetar:
 - `#NO-MODIFY: DATA LOAD`  
    🔒 **No modifiques** el contenido de esta celda.

  - `#NO-MODIFY: VARIABLE NAME`  
    ✏️ Puedes modificar o añadir información **dentro de la celda**, pero **sin cambiar el nombre de la variable asignada**. No incluyas más variables de las existentes en la celda.

  - `#MODIFY: ADD INFO TO SOLVE FUNCTION`  
    🔧 Puedes modificar el **interior de la función** para resolver la tarea, pero **no cambies su nombre, la cabecera ni el `return`**.



## Imports

In [ ]:
import numpy as np
import nltk
nltk.download('stopwords')

In [ ]:
# Add your imports here
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from collections import Counter
import re
import numpy as np
import torch
import evaluate


# 🔍 Ejercicio1: Detección de profesiones en tweets

## Enunciado

En este ejercicio vamos a trabajar con un conjunto de datos procedente de medios sociales online.

Utilizaremos un subconjunto de los datos de la tarea 1 del shared task [**ProfNER**](https://temu.bsc.es/smm4h-spanish), centrada en la detección de menciones a profesiones en tweets publicados durante la pandemia del COVID-19. El objetivo original de la tarea era analizar que profesiones podrían haber sido especialmente vulnerables en el contexto de la crisis sanitaria.

Para simplificar el ejercicio, he preparado una versión reducida del dataset original. Tu tarea será entrenar un clasificador binario basado en la arquitectura Transformers, que, dado un tweet, determine si contiene una mención explícita a una profesión (etiqueta `1`) o no (etiqueta `0`).




✅ **Objetivos del ejercicio**

A lo largo de este notebook, completarás las siguientes etapas para construir un clasificador de menciones a profesiones en tweets:

1. **Análisis Exploratorio de Datos (EDA)**: Calcular estadísticas básicas del conjunto de datos (como el número de ejemplos del training set, la distribución de clases del dataset, la longitud media de los textos) o crear visualizaciones para cmprender mejor el contenido de los documentos usando wordclouds o histogramas.

2. **Selección y justificación del modelo**: Elegir un modelo del Hub de Huggingface adecuado para los datos con los que se va a trabajar y el tipo de tarea a desarrollar.

3. **Entrenamiento del clasificador**: Entrenar el modelo de forma reproducible y evaluar su rendimiento sobreel conjunto de datos de validación, incluyendo un classification score y matriz de confusion

4. **Generación de predicciones sobre el conjunte de test**: Aplicar el modelo entrenado al conjunto de test, y guardar las predicciones en un archivo `.tsv` de 2 columnas `id` y `label` separadas por tabulador

📝 **Criterios de Evaluación**

Tu trabajo será evaluado según los siguientes criterios:

| Criterio                                            | Peso  |
|-----------------------------------------------------|--------|
| 🔍 Análisis exploratorio y preprocesamiento         | 20%   |
| 🤖 Selección y justificación del modelo             | 25%   |
| 📁 Formato y validez del archivo de predicciones    | 5%    |
| ⚙️ Ejecución correcta del notebook (sin intervención) | 10%   |
| 📈 Rendimiento del modelo sobre el conjunto de test | 30%   |
| ✍️ Claridad y calidad de las explicaciones          | 10%   |



🔔 **Nota importante:**

> El rendimiento del modelo se evaluará utilizando métricas estándar como el **F1-score** sobre el conjunto de test.

> El archivo de predicciones debe respetar **estrictamente** el formato solicitado (`id` y `label`, separados por tabulador y con extensión `.tsv`).  
  ❗ Si el archivo no cumple con este formato, **el ejercicio no podrá ser evaluado en esa sección**.

> El/la estudiante con el **mayor F1-score** obtendrá la puntuación máxima en el apartado de rendimiento. El resto de calificaciones se ajustarán de forma proporcional al mejor resultado



⚙️ **Requerimientos y reglas**

- El notebook debe ejecutarse **de principio a fin sin intervención manual**.
- Si utilizas librerías que no están incluidas por defecto en Google Colab, **asegúrate de instalarlas dentro del notebook** (por ejemplo: `!pip install ...`).

- Algunas celdas incluyen identificadores especiales que indican ciertas normas que **debes** respetar:
 - `#NO-MODIFY: DATA LOAD`  
    🔒 **No modifiques** el contenido de esta celda.

  - `#NO-MODIFY: VARIABLE NAME`  
    ✏️ Puedes modificar o añadir información **dentro de la celda**, pero **sin cambiar el nombre de la variable asignada**. No incluyas más variables de las existentes en la celda.

  - `#MODIFY: ADD INFO TO SOLVE FUNCTION`  
    🔧 Puedes modificar el **interior de la función** para resolver la tarea, pero **no cambies su nombre, la cabecera ni el `return`**.


# Tu resolución (rellena las celdas marcadas)

## Obtención de datos

Descargamos los datos del [repositorio de Huggingface](https://huggingface.co/datasets/luisgasco/profner_classification_master).

In [ ]:
#NO-MODIFY: DATA LOAD
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel
dataset = load_dataset("luisgasco/profner_classification_master")

El dataset contiene tres subsets:
- **train** y **validation**: Contienen el identificador del tweet, el texto, y su etiqueta, que podrá tener valor 1, si contiene una mención de una profesión; o valor 0, si no contiene una mención de una profesión.
- **test**: El test set tambiíen contiene la información de label por un requerimiento de Huggingface, pero el contenido de esta variable es siempre "-1". Es decir que deberéis predecir nuevas etiquetas una vez hayáis entrenado el modelo utilizando el train y el validation set.

## Análisis exploratorio de datos

Para hacer el análisis exploratorio de datos, transformamos cada subset a un pandas dataframe para mayor comodidad.

In [ ]:
#NO-MODIFY: DATA LOAD
dataset_train_df = dataset["train"].to_pandas()
dataset_val_df = dataset["validation"].to_pandas()
dataset_test_df = dataset["test"].to_pandas()

**Número de documentos**

Obten con la función `get_num_docs_evaluation()` el número de documentos del dataset de training y validation.

> Recuerda incorporar la información para el cálculo dentro del a siguiente celda, sin modificar los atributos de entrada ni de salida de la función, ni su nombre.

In [ ]:
#MODIFY: ADD INFO TO SOLVE FUNCTION
def get_num_docs_evaluation(dataset_df):
  # Contamos filas = tweets
  num_docs = len(dataset_df)

  # No modifiques el return
  return num_docs


Una vez generada la función, puedes utilizarla posteriormente para calcular resultados y comentarlos

In [ ]:
# Aplica la función
print(f"Train: {get_num_docs_evaluation(dataset_train_df)} docs")
print(f"Val: {get_num_docs_evaluation(dataset_val_df)} docs")
print(f"Test: {get_num_docs_evaluation(dataset_test_df)} docs")


**Número de documentos duplicados**

Obten con la función `detect_duplicates_evaluation()` el número de documentos duplicados del dataset de training y validation.

> Recuerda incorporar la información para el cálculo dentro del a siguiente celda, sin modificar los atributos de entrada ni de salida de la función, ni su nombre.

In [ ]:
#MODIFY: ADD INFO TO SOLVE FUNCTION
def detect_duplicates_evaluation(dataset_df):
  # Buscamos textos repetidos
  num_duplicates = dataset_df.duplicated(subset=['text']).sum()

  # No modifiques el return
  return num_duplicates


Una vez generada la función, puedes utilizarla posteriormente para calcular resultados y comentarlos

In [ ]:
# Aplica la función
print(f"Duplicados train: {detect_duplicates_evaluation(dataset_train_df)}")
print(f"Duplicados val: {detect_duplicates_evaluation(dataset_val_df)}")


**Número de documentos por cada clase:**


Obten con la función `analyse_num_labels_evaluation()` para calcular el número de documentos de cada categoría en el dataset

> Recuerda incorporar la información para el cálculo dentro del a siguiente celda, sin modificar los atributos de entrada ni de salida de la función, ni su nombre.

In [ ]:
#MODIFY: ADD INFO TO SOLVE FUNCTION
def analyse_num_labels_evaluation(dataset_df):
  # Contamos positivos (1) y negativos (0)
  num_positives = (dataset_df['label'] == 1).sum()
  num_negatives = (dataset_df['label'] == 0).sum()

  # No modifiques el return
  return num_positives, num_negatives


Una vez generada la función, puedes utilizarla posteriormente para calcular resultados y comentarlos

In [ ]:
# Aplica la función
pos, neg = analyse_num_labels_evaluation(dataset_train_df)
print(f"Train -> Con profesión: {pos}, Sin profesión: {neg}")
pos, neg = analyse_num_labels_evaluation(dataset_val_df)
print(f"Val -> Con profesión: {pos}, Sin profesión: {neg}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
dataset_train_df['label'].value_counts().plot(kind='bar', ax=ax[0], color=['steelblue','coral'])
ax[0].set_title('Train')
dataset_val_df['label'].value_counts().plot(kind='bar', ax=ax[1], color=['steelblue','coral'])
ax[1].set_title('Validation')
plt.tight_layout(); plt.show()


**Distribución de la longitud de los tweet en caracteres:**

In [ ]:
dataset_train_df['len'] = dataset_train_df['text'].apply(len)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dataset_train_df['len'], bins=30, color='steelblue', alpha=0.7)
ax.axvline(dataset_train_df['len'].mean(), color='red', ls='--',
           label=f"Media: {dataset_train_df['len'].mean():.0f}")
ax.legend(); ax.set_xlabel('Caracteres'); ax.set_title('Longitud tweets (train)')
plt.show()
print(f"Media: {dataset_train_df['len'].mean():.0f}, Max: {dataset_train_df['len'].max()}")


**Análisis de contenido de los tweets**

Para ello utiliza wordclouds

In [ ]:
stop_words = set(stopwords.words('spanish'))
stop_words.update(['https','http','co','rt','que','de','en','la','el',
                   'es','los','las','por','con','para','del','una','un'])

txt1 = ' '.join(dataset_train_df[dataset_train_df['label']==1]['text'])
txt0 = ' '.join(dataset_train_df[dataset_train_df['label']==0]['text'])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(WordCloud(stopwords=stop_words, width=600, height=300, background_color='white').generate(txt1))
ax[0].set_title('CON profesión'); ax[0].axis('off')
ax[1].imshow(WordCloud(stopwords=stop_words, width=600, height=300, background_color='white').generate(txt0))
ax[1].set_title('SIN profesión'); ax[1].axis('off')
plt.tight_layout(); plt.show()


## Tokenización

El texto del dataset no está preparado para ser introducido en un modelo Transformers. Lleva a cabo el proceso de tokenización.

In [ ]:
# IMPORTS
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)


Selecciona un modelo apropiado para la tarea:

> Recuerda que en la siguiente celda sólo debes asignar un valor a model_name. No añadas más información en la celda.

### Justificación del modelo

Elegimos **BETO** (`dccuchile/bert-base-spanish-wwm-cased`):
- Preentrenado en español (idioma de los tweets)
- Whole Word Masking mejora la comprensión morfológica
- 110M params, viable en GPU T4 de Colab
- Competitivo en benchmarks NLP español


In [ ]:
#NO-MODIFY: VARIABLE NAME
model_name = 'dccuchile/bert-base-spanish-wwm-cased'


Puedes continuar con el proceso aquí:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized = dataset.map(tokenize_function, batched=True)


In [ ]:
# El dataset ya tiene la columna 'labels', solo formateamos a torch
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

train_ds = tokenized['train']
val_ds = tokenized['validation']
test_ds = tokenized['test']


## Fine-tuning

Carga el model para ser ajustado posteriormente:

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


### Configuracion training_args

Configura los parámetros de entrenamiento del modelo.


>

> Recuerda que en la siguiente celda sólo debes asignar atributos a la variable training_args. No añadas  otras variables en la celda

In [ ]:
#NO-MODIFY: VARIABLE NAME
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    seed=42,
    report_to='none',
)


### Métricas de evaluación

Define las métricas de evaluación

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels)['f1'],
        'precision': precision_metric.compute(predictions=preds, references=labels)['precision'],
        'recall': recall_metric.compute(predictions=preds, references=labels)['recall'],
    }


In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels)['f1'],
        'precision': precision_metric.compute(predictions=preds, references=labels)['precision'],
        'recall': recall_metric.compute(predictions=preds, references=labels)['recall'],
    }


In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels)['f1'],
        'precision': precision_metric.compute(predictions=preds, references=labels)['precision'],
        'recall': recall_metric.compute(predictions=preds, references=labels)['recall'],
    }


In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=preds, references=labels)['f1'],
        'precision': precision_metric.compute(predictions=preds, references=labels)['precision'],
        'recall': recall_metric.compute(predictions=preds, references=labels)['recall'],
    }


### Ajuste del modelo

Lleva a cabo el ajuste del modelo:

In [ ]:
# Fix para bug de torchvision en Colab
import datasets as _ds_config
_ds_config.config.TORCHVISION_AVAILABLE = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


## Evaluacion

Una vez llevada a cabo el entrenamiento, realiza la evaluación del modelo.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

results = trainer.evaluate()
print(results)

preds = trainer.predict(val_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=['Sin prof','Con prof']))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=['Sin','Con'], cmap='Blues')
plt.title('Confusion Matrix'); plt.show()


## Genera predicciones

Genera predicciones sobre el test set. Recuerda que el archivo que generes y adjuntes al ejercicio debe tener dos columnas:


| id         | label |
|------------|-------|
| 1234567890 | 1     |
| 1234567891 | 0     |
| 1234567892 | 0     |
| 1234567893 | 1     |

- El archivo debe estar en formato **TSV** (separado por tabuladores).
- Debe contener exactamente **dos columnas**: `id` y `label`.
- Es obligatorio incluir la **cabecera**.


In [ ]:
# Fix: las labels del test son -1, lo que causa error en CUDA
# Las ponemos a 0 temporalmente (no afecta la predicción)
test_ds = test_ds.map(lambda x: {'labels': 0})
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Predicciones sobre test
test_preds = trainer.predict(test_ds)
y_test = np.argmax(test_preds.predictions, axis=-1)

# Guardamos en formato TSV (CAMBIA el nombre con tus apellidos)
out = pd.DataFrame({'id': dataset_test_df['id'], 'label': y_test})
out.to_csv('APELLIDO1_APELLIDO2_NOMBRE_ejercicio1_predicciones.tsv', sep='\t', index=False)
print(f"Guardado: {len(out)} predicciones")
print(out['label'].value_counts())
out.head()


### Prueba de otros modelos